# 📊 ĐÁNH GIÁ, TRỰC QUAN HÓA TOÁN HỌC & SO SÁNH MÔ HÌNH (BENCHMARK)
> Notebook này cung cấp bộ công cụ trực quan hóa toàn diện:
1. **Phân tích toán học tổng hợp:** Dạng sóng $f(x)$ và đạo hàm $\frac{df(x)}{dx}$ trên miền $[-3, 3]$ với độ tương phản cao.
2. **Phân tích chi tiết từng hàm:** Hệ trục đôi (Twin-Axes) kèm công thức giải tích $\LaTeX$.
3. **Sơ đồ Training Loss:** Phân tích độ dốc và tính ổn định khi giảm hàm mất mát (Linear & Log Scale).
4. **So sánh mô hình (Benchmark):** Đường cong hội tụ Val PSNR, biểu đồ cột $\Delta$PSNR và phân tích khối **ECA Attention**.

In [1]:
# 1. KẾT NỐI GOOGLE DRIVE & DI CHUYỂN VÀO THƯ MỤC DỰ ÁN
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM3_HK3_(2025-2026)/CT282_Deep Learning/PROJECT/RIFE-MinhTri/RIFE-Project"
os.chdir(PROJECT_DIR)
print(f"✅ Đang ở thư mục dự án: {os.getcwd()}")

Mounted at /content/drive
✅ Đang ở thư mục dự án: /content/drive/MyDrive/COMPUTER SCIENCE/NAM3_HK3_(2025-2026)/CT282_Deep Learning/PROJECT/RIFE-MinhTri/RIFE-Project


---
## 🎨 PHẦN 1: SO SÁNH TỔNG HỢP CÁC HÀM KÍCH HOẠT & ĐẠO HÀM (MIỀN [-3, 3])

In [ ]:
# 2. VẼ SO SÁNH TỔNG HỢP F(X) VÀ ĐẠO HÀM DF(X)/DX (MIỀN [-3, 3])
from visualize import plot_activations_and_gradients

plot_activations_and_gradients(save_path='demo/activations_comparison.png')

---
## 🔍 PHẦN 2: CHI TIẾT TỪNG HÀM KÍCH HOẠT RIÊNG BIỆT KÈM ĐẠO HÀM (TWIN AXES)

In [ ]:
# 3. VẼ BIỂU ĐỒ CHI TIẾT TỪNG LOẠI HÀM KÈM CÔNG THỨC VÀ ĐẠO HÀM RIÊNG BIỆT
from visualize import plot_individual_activations

plot_individual_activations(save_path='demo/individual_activations.png')

---
## 📉 PHẦN 3: SƠ ĐỒ SO SÁNH TIẾN TRÌNH GIẢM TRAINING LOSS

In [ ]:
# 4. VẼ SƠ ĐỒ TRAINING LOSS (LINEAR SCALE & LOG SCALE ĐỂ PHÂN TÍCH DAO ĐỘNG)
from visualize import plot_training_loss

plot_training_loss(models_dir='trained_model', save_path='demo/training_loss_comparison.png')

---
## 📊 PHẦN 4: SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH VÀ KHỐI ECA ATTENTION

In [ ]:
# 5. TỰ ĐỘNG QUÉT TRAINED_MODEL/ VÀ VẼ BIỂU ĐỒ HỘI TỤ PSNR & BEST PSNR
from visualize import plot_model_comparisons
from IPython.display import display

df_summary = plot_model_comparisons(models_dir='trained_model', save_path='demo/benchmark_models_comparison.png')

if df_summary is not None:
    print("\n📋 BẢNG TỔNG KẾT SO SÁNH CHỈ SỐ ΔPSNR SO VỚI BASELINE GỐC:")
    display(df_summary)

---
## 📈 PHẦN 5: BẢNG SO SÁNH TÁC ĐỘNG TĂNG TRƯỞNG KHI THÊM ECA ATTENTION

In [ ]:
# 6. SO SÁNH HIỆU QUẢ TRƯỚC VÀ SAU KHI THÊM ECA TRÊN TỪNG HÀM KÍCH HOẠT
import os
import json
import pandas as pd
from IPython.display import display

models_dir = 'trained_model'
acts = ['prelu', 'gelu', 'silu', 'smooth_prelu', 'optimized_smooth_prelu', 'soft_clamp_relu', 'soft_clamp_silu']
eca_comparison = []

for act in acts:
    base_file = os.path.join(models_dir, f'baseline_{act}', 'experiment_results.json')
    eca_file = os.path.join(models_dir, f'modify_eca_{act}', 'experiment_results.json')

    base_psnr = None
    eca_psnr = None

    if os.path.exists(base_file):
        with open(base_file) as f:
            d = json.load(f)
            base_psnr = max([x['val_psnr'] for x in d]) if d else None

    if os.path.exists(eca_file):
        with open(eca_file) as f:
            d = json.load(f)
            eca_psnr = max([x['val_psnr'] for x in d]) if d else None

    if base_psnr is not None or eca_psnr is not None:
        delta_eca = (eca_psnr - base_psnr) if (eca_psnr and base_psnr) else None
        eca_comparison.append({
            'Hàm Kích Hoạt': act.upper(),
            'Baseline (Không ECA)': f'{base_psnr:.2f} dB' if base_psnr else 'Chưa train',
            'Modify (+ Khối ECA)': f'{eca_psnr:.2f} dB' if eca_psnr else 'Chưa train',
            'Độ Tăng Trưởng (+ ΔECA)': f'{delta_eca:+.2f} dB' if delta_eca is not None else 'N/A'
        })

if eca_comparison:
    df_eca = pd.DataFrame(eca_comparison)
    print('✨ BẢNG SO SÁNH TÁC ĐỘNG TĂNG TRƯỞNG KHI THÊM ECA ATTENTION:')
    display(df_eca)
else:
    print('⚠️ Hãy train cả mô hình baseline_* và modify_eca_* để xem bảng so sánh tăng trưởng ECA.')